In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive/Medicine-Delivery-Optimization/data/raw

delivery_partners.csv  inventory.csv  orders.csv


In [5]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Medicine-Delivery-Optimization/data/raw/orders.csv")
df.head()

,order_id,city,order_time,dispatch_time,delivery_time,warehouse_id,partner_id,medicine_type,order_value,sla_breach
0,1,Mumbai,2024-01-24 06:16:00,2024-01-24 06:46:35.276951,2024-01-24 13:02:05.570877,202,30,acute,818.80,True
1,2,Chennai,2024-02-20 06:49:00,2024-02-20 08:05:11.295583,2024-02-20 11:05:35.248481,401,26,chronic,749.75,False
2,3,Bangalore,2024-01-28 14:28:00,2024-01-28 15:00:23.162199,2024-01-28 19:38:03.538299,102,22,chronic,469.43,False
3,4,Delhi,2024-02-16 06:08:00,2024-02-16 06:45:40.231684,2024-02-16 10:22:20.845151,301,117,acute,1880.74,False
4,5,Chennai,2024-02-06 18:30:00,2024-02-06 19:57:57.426540,2024-02-06 23:13:49.212603,402,150,acute,1014.30,False


In [6]:
df["order_time"] = pd.to_datetime(df["order_time"])
df["dispatch_time"] = pd.to_datetime(df["dispatch_time"])
df["delivery_time"] = pd.to_datetime(df["delivery_time"])

In [7]:
invalid_dispatch = df[df["dispatch_time"] < df["order_time"]]
invalid_delivery = df[df["delivery_time"] < df["dispatch_time"]]

print("Invalid dispatch:", len(invalid_dispatch))
print("Invalid delivery:", len(invalid_delivery))

Invalid dispatch: 0
Invalid delivery: 0


In [8]:
df.isnull().sum()

,0
order_id,0
city,0
order_time,0
dispatch_time,0
delivery_time,0
warehouse_id,0
partner_id,0
medicine_type,0
order_value,0
sla_breach,0


In [9]:
df["delivery_time_hours"] = (
    (df["delivery_time"] - df["order_time"]).dt.total_seconds() / 3600
)

In [10]:
df["dispatch_delay_hours"] = (
    (df["dispatch_time"] - df["order_time"]).dt.total_seconds() / 3600
)

In [11]:
df["delivery_delay_hours"] = (
    (df["delivery_time"] - df["dispatch_time"]).dt.total_seconds() / 3600
)

In [12]:
df["order_hour"] = df["order_time"].dt.hour

In [13]:
df["is_late_night"] = df["order_hour"].apply(
    lambda x: 1 if (x >= 22 or x <= 5) else 0
)

In [14]:
df["is_high_value"] = df["order_value"].apply(
    lambda x: 1 if x > 1000 else 0
)

In [15]:
def delay_category(row):
    if row["dispatch_delay_hours"] > 2:
        return "Warehouse Delay"
    elif row["delivery_delay_hours"] > 3:
        return "Delivery Delay"
    else:
        return "On Time"

df["delay_type"] = df.apply(delay_category, axis=1)

In [16]:
print(df[[
    "delivery_time_hours",
    "dispatch_delay_hours",
    "delivery_delay_hours"
]].describe())

       delivery_time_hours  dispatch_delay_hours  delivery_delay_hours
count         50000.000000          50000.000000          50000.000000
mean              5.211237              1.421030              3.790207
std               1.213842              0.848842              0.880120
min               2.713670              0.500022              2.200010
25%               4.342615              0.838756              3.126579
50%               5.091473              1.172992              3.776809
75%               5.926996              1.551211              4.367435
max              11.171781              6.201217              7.060644


In [19]:
df.to_csv("/content/drive/MyDrive/Medicine-Delivery-Optimization/data/processed/orders_clean.csv", index=False)